In [ ]:
import os
from pathlib import Path
import torch
import numpy as np
import cv2
import glob
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt



# **Functions**

In [ ]:
NOISE_TYPE='poiss'

def add_noise(image, noise_level):
    """Add synthetic noise to image"""
    if NOISE_TYPE == 'gauss':
        noisy = image+ torch.randn_like(image) * (noise_level/255)
        return torch.clamp(noisy, 0, 1)

    if NOISE_TYPE == 'poiss':
        return torch.poisson(noise_level * image) / noise_level

    else:
      raise ValueError(f"Invalid noise type: {NOISE_TYPE}")


def adjust_gamma(image, gamma=1.0):
	# build a lookup table mapping the pixel values [0, 255] to
	# their adjusted gamma values
	invGamma = 1.0 / gamma
	table = np.array([((i / 255.0) ** invGamma) * 255
		for i in np.arange(0, 256)]).astype("uint8")
	# apply gamma correction using the lookup table
	return cv2.LUT(image, table)

In [ ]:
# Cell 4: Helper functions
def load_image_as_tensor(image_path):
    img_np = cv2.imread(str(image_path))
    img_tensor = torch.from_numpy(img_np).float() / 255.0
    img_tensor = img_tensor.permute(2, 0, 1).unsqueeze(0)
    return img_tensor

def save_tensor_as_image(tensor, save_path):
    img_np = (tensor.squeeze(0).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    cv2.imwrite(str(save_path), img_np)



# **Gamma on V-Channel Only**


In [ ]:
#  Robust Color Conversion Functions
def rgb_to_hsv_torch(image: torch.Tensor) -> torch.Tensor:
    """
    Convert RGB to HSV using PyTorch primitive operations for stability.
    """
    r, g, b = image[:, 0, :, :], image[:, 1, :, :], image[:, 2, :, :]

    max_val, _ = image.max(dim=1)
    min_val, _ = image.min(dim=1)
    diff = max_val - min_val

    # Saturation
    s = torch.zeros_like(max_val)
    mask_s = max_val > 0
    s[mask_s] = diff[mask_s] / max_val[mask_s]

    # Hue
    h = torch.zeros_like(max_val)
    mask_diff = diff > 0

    mask_r = (max_val == r) & mask_diff
    h[mask_r] = (g[mask_r] - b[mask_r]) / diff[mask_r]

    mask_g = (max_val == g) & mask_diff
    h[mask_g] = 2.0 + (b[mask_g] - r[mask_g]) / diff[mask_g]

    mask_b = (max_val == b) & mask_diff
    h[mask_b] = 4.0 + (r[mask_b] - g[mask_b]) / diff[mask_b]

    h = (h / 6.0) % 1.0

    # Value
    v = max_val

    return torch.stack([h, s, v], dim=1)

def hsv_to_rgb_torch(image: torch.Tensor) -> torch.Tensor:
    h, s, v = image[:, 0, :, :], image[:, 1, :, :], image[:, 2, :, :]

    i = (h * 6.0).floor()
    f = (h * 6.0) - i
    p = v * (1.0 - s)
    q = v * (1.0 - s * f)
    t = v * (1.0 - s * (1.0 - f))

    i = i % 6

    rgb = torch.zeros_like(image)

    # Stack condition masks
    mask_0 = (i == 0)
    mask_1 = (i == 1)
    mask_2 = (i == 2)
    mask_3 = (i == 3)
    mask_4 = (i == 4)
    mask_5 = (i == 5)

    # R channel
    rgb[:, 0, :, :][mask_0] = v[mask_0]
    rgb[:, 0, :, :][mask_1] = q[mask_1]
    rgb[:, 0, :, :][mask_2] = p[mask_2]
    rgb[:, 0, :, :][mask_3] = p[mask_3]
    rgb[:, 0, :, :][mask_4] = t[mask_4]
    rgb[:, 0, :, :][mask_5] = v[mask_5]

    # G channel
    rgb[:, 1, :, :][mask_0] = t[mask_0]
    rgb[:, 1, :, :][mask_1] = v[mask_1]
    rgb[:, 1, :, :][mask_2] = v[mask_2]
    rgb[:, 1, :, :][mask_3] = q[mask_3]
    rgb[:, 1, :, :][mask_4] = p[mask_4]
    rgb[:, 1, :, :][mask_5] = p[mask_5]

    # B channel
    rgb[:, 2, :, :][mask_0] = p[mask_0]
    rgb[:, 2, :, :][mask_1] = p[mask_1]
    rgb[:, 2, :, :][mask_2] = t[mask_2]
    rgb[:, 2, :, :][mask_3] = v[mask_3]
    rgb[:, 2, :, :][mask_4] = v[mask_4]
    rgb[:, 2, :, :][mask_5] = q[mask_5]

    return torch.clamp(rgb, 0, 1)

def adjust_gamma_hsv(image_tensor, gamma):

    # 1. RGB → HSV
    hsv = rgb_to_hsv_torch(image_tensor)

    # 2. Apply gamma ONLY to V channel
    hsv[:, 2, :, :] = torch.pow(hsv[:, 2, :, :], gamma)

    # 3. HSV → RGB
    return hsv_to_rgb_torch(hsv)

In [ ]:
INPUT_FOLDER = Path("gt")
BASE_FOLDER = Path("noisy_images/synthetic_dataset_V")
BASE_FOLDER.mkdir(parents=True, exist_ok=True)
GAMMA_VALUES = [2, 3]
NOISE_LEVELS = [10, 20, 30, 40, 50]

In [ ]:
# Generate Synthetic Dataset
print("=" * 60)
print("SYNTHETIC DATASET: GAMMA ON V ONLY")
print("=" * 60)

input_folder = Path(INPUT_FOLDER)
Base_folder = Path(BASE_FOLDER)
all_images = list(input_folder.glob("*.png"))

print(f"Input folder: {input_folder}")
print(f"Output folder: {Base_folder}")
print(f"Found {len(all_images)} images")
print(f"Gamma values: {GAMMA_VALUES}")
print(f"Noise levels: {NOISE_LEVELS}")
print("-" * 60)

for image_path in all_images:
    original_name = image_path.name
    image_tensor = load_image_as_tensor(str(image_path))
    
    for gamma in GAMMA_VALUES:
        adjusted = adjust_gamma_hsv(image_tensor, gamma)
        output_folder_gamma = Base_folder / f"G{gamma}"
        output_folder_gamma.mkdir(parents=True, exist_ok=True)

        
        for noise_level in NOISE_LEVELS:
            noisy = add_noise(adjusted, noise_level)
            
            noise_folder = output_folder_gamma / f"noise_{noise_level}"
            noise_folder.mkdir(parents=True, exist_ok=True)
            
            
            out_path = noise_folder / original_name
            
            if out_path.exists():
                print(f"Skipping {original_name} (already exists)")
                continue

            save_tensor_as_image(noisy, out_path)
            print(f"Saved: {out_path}")

print("-" * 60)
print("Synthetic Dataset complete!")
print(f"Output: {noise_folder}")
print("=" * 60)

# ***Apply to whole Image***

In [ ]:
INPUT_FOLDER = Path("gt")
BASE_FOLDER = Path("noisy_images/synth_WholeImage")
BASE_FOLDER.mkdir(parents=True, exist_ok=True)
GAMMA_VALUES = [2, 3]
NOISE_LEVELS = [10, 20, 30, 40, 50]

In [ ]:
# Generate Whole Image Dataset
print("=" * 60)
print("WHOLE IMAGE DATASET: OPENCV GAMMA CORRECTION")
print("=" * 60)

input_folder = Path(INPUT_FOLDER)
base_folder = Path(BASE_FOLDER)
all_images = list(input_folder.glob("*.png"))

print(f"Input folder: {input_folder}")
print(f"Output folder: {base_folder}")
print(f"Found {len(all_images)} images")
print(f"Gamma values: {GAMMA_VALUES}")
print(f"Noise levels: {NOISE_LEVELS}")
print("-" * 60)

for image_path in all_images:
    original_name = image_path.name
    
    # Load as numpy (BGR)
    img_np = cv2.imread(str(image_path))
    
    for gamma in GAMMA_VALUES:
        # Apply gamma correction using OpenCV
        adjusted_np = adjust_gamma(img_np, 1/gamma)
        output_folder_gamma = base_folder / f"G{gamma}"
        output_folder_gamma.mkdir(parents=True, exist_ok=True)
        
        # Convert to tensor for noise addition
        adjusted_tensor = torch.from_numpy(adjusted_np).float() / 255.0
        adjusted_tensor = adjusted_tensor.permute(2, 0, 1).unsqueeze(0)
        
        for noise_level in NOISE_LEVELS:
            noisy = add_noise(adjusted_tensor, noise_level)
            
            noise_folder = output_folder_gamma / f"noise_{noise_level}"
            noise_folder.mkdir(parents=True, exist_ok=True)
            

            out_path = noise_folder / original_name
            
            if out_path.exists():
                print(f"Skipping {original_name} (already exists)")
                continue
            
            save_tensor_as_image(noisy, out_path)
            print(f"Saved: {out_path}")

print("-" * 60)
print("Whole Image Dataset complete!")
print(f"Output: {noise_folder}")
print("=" * 60)